# NFL Big Data Bowl 2027 — Initial Data Exploration

**Status:** project setup; no research results yet.

This notebook inspects the competition files and a bounded CSV sample before a research question is selected. Run cells in order. Download the competition data from Kaggle and extract it into `data/raw/` for local work.

On Kaggle, attach the competition data; this notebook uses `/kaggle/input` when available. No API token is needed inside the notebook.

**Important:** the preview is a sample. It does not establish full-data coverage, complete player counts, or football performance conclusions.


In [ ]:
from pathlib import Path
import sys
import importlib.metadata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Locate the input files

The local project root is detected from `requirements.txt` and `notebooks/`, so the notebook works when started from the project root or the notebook directory. The inventory lists CSV and Parquet files; the starter sample loader below reads CSV only.


In [ ]:
current_dir = Path.cwd().resolve()
project_candidates = [current_dir, *current_dir.parents]
PROJECT_ROOT = next(
    (path for path in project_candidates
     if (path / "requirements.txt").is_file() and (path / "notebooks").is_dir()),
    current_dir,
)

KAGGLE_INPUT = Path("/kaggle/input")
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.is_dir() else PROJECT_ROOT / "data" / "raw"

input_files = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in {".csv", ".parquet"}
)
inventory = pd.DataFrame(
    [{"file": str(path.relative_to(DATA_DIR)),
      "format": path.suffix.lower().lstrip("."),
      "size_mb": round(path.stat().st_size / 1024**2, 2)}
     for path in input_files],
    columns=["file", "format", "size_mb"],
)

print("Data directory:", DATA_DIR)
print("Input files found:", len(input_files))
display(inventory)

if not input_files:
    print("Next: download and extract the competition files into data/raw/, then rerun this notebook.")


## 2. Read one CSV sample

Leave `SELECTED_FILE = None` to inspect the first CSV in the sorted inventory, or replace it with a relative path from the inventory above. For example, use `"folder/actual_filename.csv"` only after confirming that file exists.

`MAX_ROWS` limits memory use. The first rows of a file are not necessarily a representative sample of all players, seasons, or drills.


In [ ]:
SELECTED_FILE = None
MAX_ROWS = 50_000

csv_files = [path for path in input_files if path.suffix.lower() == ".csv"]
selected_path = None
df = None

if SELECTED_FILE is not None:
    selected_path = DATA_DIR / SELECTED_FILE
    if not selected_path.is_file() or selected_path.suffix.lower() != ".csv":
        raise ValueError("SELECTED_FILE must name an existing CSV from the inventory.")
elif csv_files:
    selected_path = csv_files[0]

if selected_path is None:
    print("No CSV selected. Add competition CSV files to continue.")
else:
    df = pd.read_csv(selected_path, nrows=MAX_ROWS, low_memory=False)
    print("Selected file:", selected_path.relative_to(DATA_DIR))
    print("Sample shape:", df.shape)
    print("Row limit:", MAX_ROWS)
    display(df.head())


## 3. Inspect schema and missing values

Use the actual data dictionary to determine meanings and units. A numeric column may be an identifier, and inferred pandas dtypes are not a substitute for that dictionary.


In [ ]:
if df is not None:
    schema = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "sample_non_null": df.notna().sum(),
        "sample_missing_pct": df.isna().mean().mul(100).round(2),
        "sample_unique_values": df.nunique(dropna=True),
    })
    schema.index.name = "column"
    display(schema)
    print("Exact duplicate rows in sample:", int(df.duplicated().sum()))
    if len(df) > 0:
        display(df.describe(include="all").T)
    else:
        print("The selected CSV has a header but no data rows.")
else:
    print("Load a CSV sample before inspecting its schema.")


## 4. Inspect categorical values

The following preview is limited to ten text/category columns and five values per column. Use it to recognize labels such as position groups, drills, or events if they are present in the actual schema.


In [ ]:
if df is not None:
    categorical_columns = df.select_dtypes(include=["object", "string", "category"]).columns[:10]
    for column in categorical_columns:
        print(column)
        display(df[column].value_counts(dropna=False).head(5))
else:
    print("Load a CSV sample before inspecting categorical values.")


## 5. Check candidate identifiers

After inspecting the schema, set `KEY_COLUMNS` to columns that you expect to identify a record. Repeated player IDs are normal in tracking data; they do not by themselves indicate duplicates. Do not delete rows just because this check finds repeated keys.


In [ ]:
KEY_COLUMNS = []

if df is None:
    print("Load a CSV sample before checking keys.")
elif not KEY_COLUMNS:
    print("Choose candidate key columns after reading the data dictionary.")
else:
    absent_columns = [column for column in KEY_COLUMNS if column not in df.columns]
    if absent_columns:
        raise ValueError(f"Unknown key columns: {absent_columns}")
    print("Repeated candidate keys in sample:", int(df.duplicated(subset=KEY_COLUMNS).sum()))
    print("Rows with missing key values:", int(df[KEY_COLUMNS].isna().any(axis=1).sum()))


## 6. Plot a meaningful numeric feature

Set `PLOT_COLUMN` to a numeric feature with a known meaning and unit. Avoid plotting an ID as if it were a measurement. This cell deliberately leaves the feature choice to you after inspection.


In [ ]:
PLOT_COLUMN = None

if df is None:
    print("Load a CSV sample before plotting.")
elif PLOT_COLUMN is None:
    print("Choose PLOT_COLUMN after confirming its meaning and unit.")
else:
    if PLOT_COLUMN not in df.columns or not pd.api.types.is_numeric_dtype(df[PLOT_COLUMN]):
        raise ValueError("PLOT_COLUMN must be an existing numeric column.")
    values = df[PLOT_COLUMN].dropna()
    if values.empty:
        print("No non-missing values to plot in this sample.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4))
        sns.histplot(values, bins=40, ax=ax)
        ax.set_title(f"Sample distribution: {PLOT_COLUMN}")
        ax.set_xlabel(PLOT_COLUMN)
        ax.set_ylabel("Sample row count")
        fig.tight_layout()
        plt.show()


## 7. Record what you learned

Complete `reports/research_notes.md` using the actual dataset:

- What does one row represent in each file?
- What are the coordinate, speed, acceleration, and time units?
- Which columns identify players, drills, games, plays, and time/frame records?
- How can Combine and regular-season records be linked?
- Which positions, seasons, and drills are covered?
- Do missing records reflect non-participation, filtering, or another process?
- Which narrow football question is feasible with these data?

Before modeling, define the outcome, baseline, and validation split. Avoid allowing the same player's records into both training and validation when the aim is to generalize to unseen players.


## 8. Inspect the environment

This table records package versions available to the current kernel. On your laptop, save a complete dependency snapshot from the working virtual environment with `python -m pip freeze > requirements-lock.txt` and commit it when experiments begin.


In [ ]:
environment_rows = []
for package in ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "pyarrow", "jupyterlab", "ipykernel"]:
    try:
        installed_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_version = "not installed in this kernel environment"
    environment_rows.append({"package": package, "version": installed_version})
display(pd.DataFrame(environment_rows))
